# InstaNovo+ evaluation — Ecoli_EV_2 (held-out test)

Mirrors `instanovo_colab_evaluate.ipynb` but for the diffusion model.
Runs `instanovo diffusion predict --evaluation` against
`Ecoli_EV_2.instanovo.annotated.mgf` with **both** the pretrained and
fine-tuned InstaNovo+ checkpoints. Saves both prediction CSVs to Drive,
and prints peptide-level exact-match rates side-by-side as a quick sanity
check.

**Refinement coupling:** InstaNovo+ refines InstaNovo (transformer)
predictions. Pretrained refines pretrained, finetuned refines finetuned —
we use the InstaNovo eval CSVs (produced by `instanovo_colab_evaluate.ipynb`)
as the `refinement_path` input.

**Inputs (must exist on Drive):**
- `MyDrive/DL-Project/data_mgf_annotated/ecoli/Ecoli_EV_2.instanovo.annotated.mgf`  (UNIMOD notation)
- `MyDrive/DL-Project/bin/instanovoplus/instanovoplus-v1.1.0.ckpt`             (pretrained)
- `MyDrive/DL-Project/model_finetune/instanovoplus/model_best.ckpt`           (fine-tuned, lowest-valid_loss state)
- `MyDrive/DL-Project/result_finetune_annotated/instanovo/Ecoli_EV_2.pretrained.csv`   (InstaNovo pretrained predictions on this MGF)
- `MyDrive/DL-Project/result_finetune_annotated/instanovo/Ecoli_EV_2.finetuned.csv`    (InstaNovo finetuned predictions on this MGF)

**Outputs (written to Drive):**
- `MyDrive/DL-Project/result_finetune_annotated/instanovoplus/Ecoli_EV_2.pretrained.csv`
- `MyDrive/DL-Project/result_finetune_annotated/instanovoplus/Ecoli_EV_2.finetuned.csv`

These go under `result_finetune_annotated/` (not `result_finetune/`) to
keep eval outputs (against annotated MGF) cleanly separated from
inference outputs (which go to `result_finetune/<tool>/ecoli/...` for
the FDR pipeline).

**Note on `model_best.ckpt` availability:** the InstaNovo+ diffusion parent
config doesn't set `checkpoint_metric` / `_mode` by default — we add them
in `config/finetune/instanovoplus.yaml` so the trainer writes both
`model_best.ckpt` (lowest valid_loss) and `model_latest.ckpt` (end of
training). If your existing run only has `model_latest.ckpt`, re-run the
finetune notebook with the updated YAML to get model_best.

**Note on residues:** Both InstaNovo+ ckpts (v1.1.0 pretrained and the
fine-tuned `model_best.ckpt`) store residues flat at `ckpt["residues"]`,
which is what `InstaNovoPlus.load()` (called by `diffusion predict`)
expects. The patch cell below is idempotent — flattens if the ckpt is
in the trainer's nested form, no-op if already flat.


In [23]:
!nvidia-smi

Sat May  2 18:00:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Install dependencies

In [24]:
try:
  import instanovo
  !instanovo version
except ImportError:
  !pip install "instanovo[cu126]>=1.2.2" pyopenms-viz
  print('Installation complete. Restarting runtime to apply changes...')
  import os
  os.kill(os.getpid(), 9)

┏━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Package    ┃ Version ┃
┡━━━━━━━━━━━━╇━━━━━━━━━┩
│ InstaNovo  │ 1.2.2   │
│ InstaNovo+ │ 1.2.2   │
│ NumPy      │ 2.2.6   │
│ PyTorch    │ 2.8.0   │
└────────────┴─────────┘


## Sync inputs from Drive

In [25]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/data_mgf_annotated/ecoli', exist_ok=True)
os.makedirs('/content/bin/instanovoplus', exist_ok=True)
os.makedirs('/content/model_finetune/instanovoplus', exist_ok=True)
os.makedirs('/content/result_finetune_annotated/instanovo', exist_ok=True)
os.makedirs('/content/result_finetune_annotated/instanovoplus', exist_ok=True)

# Annotated test MGF (UNIMOD notation — produced by annotate_mgf.py --notation unimod)
!cp /content/drive/MyDrive/DL-Project/data_mgf_annotated/ecoli/Ecoli_EV_2.instanovo.annotated.mgf /content/data_mgf_annotated/ecoli/

# Pretrained InstaNovo+ ckpt
!cp /content/drive/MyDrive/DL-Project/bin/instanovoplus/instanovoplus-v1.1.0.ckpt /content/bin/instanovoplus/instanovoplus-v1.1.0.ckpt

# Fine-tuned InstaNovo+ ckpt (output of instanovoplus_colab_finetune.ipynb)
!cp /content/drive/MyDrive/DL-Project/model_finetune/instanovoplus/model_best.ckpt /content/model_finetune/instanovoplus/model_best.ckpt

# InstaNovo (transformer) eval CSVs — used as refinement_path. Produced
# by instanovo_colab_evaluate.ipynb's predict cells.
!cp /content/drive/MyDrive/DL-Project/result_finetune_annotated/instanovo/Ecoli_EV_2.pretrained.csv /content/result_finetune_annotated/instanovo/
!cp /content/drive/MyDrive/DL-Project/result_finetune_annotated/instanovo/Ecoli_EV_2.finetuned.csv  /content/result_finetune_annotated/instanovo/

print('--- inputs ---')
!ls -lh /content/data_mgf_annotated/ecoli/
!ls -lh /content/bin/instanovoplus/
!ls -lh /content/model_finetune/instanovoplus/
!ls -lh /content/result_finetune_annotated/instanovo/

# Sanity check: SEQ= lines should already be in UNIMOD form on disk.
print('\n--- first SEQ= entries (expect UNIMOD bracket form) ---')
!grep -m3 '^SEQ=' /content/data_mgf_annotated/ecoli/Ecoli_EV_2.instanovo.annotated.mgf


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- inputs ---
total 66M
-rw------- 1 root root  27M May  2 17:52 Ecoli_EV_1.instanovo.train.mgf
-rw------- 1 root root 5.0M May  2 17:52 Ecoli_EV_1.instanovo.val.mgf
-rw------- 1 root root  34M May  2 18:00 Ecoli_EV_2.instanovo.annotated.mgf
total 659M
-rw------- 1 root root 659M May  2 18:00 instanovoplus-v1.1.0.ckpt
total 1.3G
-rw-r--r-- 1 root root 659M May  2 18:00 model_best.ckpt
-rw-r--r-- 1 root root 659M May  2 17:57 model_latest.ckpt
total 5.8M
-rw-r--r-- 1 root root 2.9M May  2 18:00 Ecoli_EV_2.finetuned.csv
-rw-r--r-- 1 root root 2.9M May  2 18:00 Ecoli_EV_2.pretrained.csv

--- first SEQ= entries (expect UNIMOD bracket form) ---
SEQ=VSHGC[UNIMOD:4]VR
SEQ=SFSHQAGASSK
SEQ=IGHTVER


## Patch ckpts to predict-CLI's expected residue format

Same train/predict asymmetry as the InstaNovo (transformer) eval. The
diffusion train code reads `ckpt["residues"]["residues"]` (nested), the
diffusion predict code reads `ckpt["residues"]` directly (flat).
Empirically, both pretrained and trainer-saved ckpts arrive flat from
Drive, so this cell is usually a no-op — but it's idempotent and protects
against accidentally syncing a wrapped ckpt.

In [26]:
import torch
from omegaconf import DictConfig, OmegaConf

def _is_wrapped(d):
    return hasattr(d, 'keys') and list(d.keys()) == ['residues']

def _to_plain_dict(x):
    if isinstance(x, DictConfig):
        return OmegaConf.to_container(x, resolve=True)
    return dict(x) if hasattr(x, 'keys') else x

def normalize_residues(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    r = ckpt.get('residues')
    if r is None:
        print(f'  {ckpt_path}: no residues key, skipping.')
        return
    if _is_wrapped(r):
        inner = _to_plain_dict(r['residues'] if hasattr(r, '__getitem__') else r.residues)
        print(f'  {ckpt_path}: flattening {len(inner)} residues out of nested wrapper.')
        ckpt['residues'] = inner
        torch.save(ckpt, ckpt_path)
    else:
        flat = _to_plain_dict(r)
        n = len(flat) if hasattr(flat, '__len__') else '?'
        print(f'  {ckpt_path}: already flat ({n} entries), no change.')

print('Normalizing InstaNovo+ ckpts to flat residues format (predict-CLI compatible)...')
normalize_residues('/content/bin/instanovoplus/instanovoplus-v1.1.0.ckpt')
normalize_residues('/content/model_finetune/instanovoplus/model_best.ckpt')
print('Done.')


Normalizing InstaNovo+ ckpts to flat residues format (predict-CLI compatible)...
  /content/bin/instanovoplus/instanovoplus-v1.1.0.ckpt: already flat (30 entries), no change.
  /content/model_finetune/instanovoplus/model_best.ckpt: already flat (30 entries), no change.
Done.


## Run prediction — pretrained checkpoint

`--evaluation` mode (vs the default `--denovo`) compares predictions
against the SEQ= labels in the annotated MGF and prints peptide / AA
precision at the end of the run.

`refinement_path=...` is a Hydra config override (not a CLI flag) —
passed as `key=value`. It points at the pretrained-InstaNovo predictions
on the same annotated MGF, which the diffusion model refines.

In [27]:
!instanovo diffusion predict \
    --evaluation \
    --data-path  /content/data_mgf_annotated/ecoli/Ecoli_EV_2.instanovo.annotated.mgf \
    --output-path /content/result_finetune_annotated/instanovoplus/Ecoli_EV_2.pretrained.csv \
    --instanovo-plus-model /content/bin/instanovoplus/instanovoplus-v1.1.0.ckpt \
    refinement_path=/content/result_finetune_annotated/instanovo/Ecoli_EV_2.pretrained.csv \
    num_workers=4 \
    batch_size=512


[05/02/26 18:00:23] INFO     Initializing InstaNovo+ inference.                                                                                                                
[05/02/26 18:00:26] INFO     NumExpr defaulting to 12 threads.                                                                                                                 
2026-05-02 18:00:28.782845: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 18:00:28.851730: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-05-02 18:00

## Run prediction — fine-tuned checkpoint

Same MGF and flags, just swap the model path to `model_best.ckpt` and
use the FINETUNED InstaNovo predictions as refinement (so the comparison
is finetuned-on-finetuned vs pretrained-on-pretrained — both stages of
the pipeline are fine-tuned together).

In [28]:
!instanovo diffusion predict \
    --evaluation \
    --data-path  /content/data_mgf_annotated/ecoli/Ecoli_EV_2.instanovo.annotated.mgf \
    --output-path /content/result_finetune_annotated/instanovoplus/Ecoli_EV_2.finetuned.csv \
    --instanovo-plus-model /content/model_finetune/instanovoplus/model_best.ckpt \
    refinement_path=/content/result_finetune_annotated/instanovo/Ecoli_EV_2.finetuned.csv \
    num_workers=4 \
    batch_size=512


[05/02/26 18:01:37] INFO     Initializing InstaNovo+ inference.                                                                                                                
[05/02/26 18:01:40] INFO     NumExpr defaulting to 12 threads.                                                                                                                 
2026-05-02 18:01:42.481508: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 18:01:42.551829: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-05-02 18:01

## Compare predictions to SEQ= labels (sanity check)

InstaNovo+'s `--evaluation` mode prints peptide / AA precision at the end
of each run above. This cell duplicates the peptide-level number locally
as a sanity check, joining each model's `predictions` column to the SEQ=
labels in the MGF on `scan_number` (the 0-based file index InstaNovo+
emits).

**Two caveats** (same as InstaNovo evaluate):

1. **Strict string match counts I/L disagreements as wrong** even though
   mass spec can't distinguish them. The real biological accuracy is a few
   points higher.
2. **This is one number, not the full picture.** Downstream FDR-yield
   comparisons via the prediction CSVs we just wrote (Jetson-side
   pipeline) are the wastewater answer.

In [29]:
import pandas as pd

MGF = '/content/data_mgf_annotated/ecoli/Ecoli_EV_2.instanovo.annotated.mgf'

def load_seqs_from_mgf(mgf_path):
    """Return {scan_number (0-based file index): SEQ string} for every spectrum."""
    seqs = {}
    idx = -1
    with open(mgf_path) as f:
        for line in f:
            if line.startswith('BEGIN IONS'):
                idx += 1
            elif line.startswith('SEQ='):
                seqs[idx] = line.split('=', 1)[1].strip()
    return seqs

def compare(pred_csv, label, seqs):
    df = pd.read_csv(pred_csv)
    df['gt'] = df['scan_number'].map(seqs)
    labelled = df.dropna(subset=['gt'])
    exact = (labelled['predictions'] == labelled['gt']).sum()
    rate = exact / len(labelled) if len(labelled) else 0
    print(f'{label:<14} {len(df):>5} predictions   '
          f'{len(labelled):>5} labelled   '
          f'{exact:>5} exact   '
          f'{rate:>6.2%}')
    return labelled

seqs = load_seqs_from_mgf(MGF)
print(f'Ground-truth SEQ= labels in MGF: {len(seqs)}\n')
print(f'{"":<14} {"preds":>5}              {"in GT":>5}     {"exact":>5}     rate')
print('-' * 70)
pre  = compare('/content/result_finetune_annotated/instanovoplus/Ecoli_EV_2.pretrained.csv', 'pretrained',  seqs)
ft   = compare('/content/result_finetune_annotated/instanovoplus/Ecoli_EV_2.finetuned.csv',  'finetuned',   seqs)

# Side-by-side spot check for the first 8 GT scans
print('\nSide-by-side sample (first 8 GT-labelled scans):')
common = set(pre['scan_number']) & set(ft['scan_number'])
sample_scans = sorted(common)[:8]
pre_idx = pre.set_index('scan_number')
ft_idx  = ft.set_index('scan_number')
print(f'{"scan":>5}  {"GT":<32}  {"pretrained":<32}  {"finetuned":<32}')
print('-' * 110)
for s in sample_scans:
    gt   = pre_idx.loc[s, 'gt'][:32]
    p    = pre_idx.loc[s, 'predictions'][:32]
    f    = ft_idx.loc[s, 'predictions'][:32]
    print(f'{s:>5}  {gt:<32}  {p:<32}  {f:<32}')


Ground-truth SEQ= labels in MGF: 1273

               preds              in GT     exact     rate
----------------------------------------------------------------------
pretrained      1273 predictions    1273 labelled     667 exact   52.40%
finetuned       1273 predictions    1273 labelled     881 exact   69.21%

Side-by-side sample (first 8 GT-labelled scans):
 scan  GT                                pretrained                        finetuned                       
--------------------------------------------------------------------------------------------------------------
    0  VSHGC[UNIMOD:4]VR                 VSHGC[UNIMOD:4]VR                 VSHGC[UNIMOD:4]VR               
    1  SFSHQAGASSK                       SFSHQAGASSK                       SFSHQAGASSK                     
    2  IGHTVER                           LGHTVER                           LGHTVER                         
    3  IEQAPGQHGAR                       LEQAPGQHGAR                       IEQAPGQHGAR      

## Push prediction CSVs to Drive

Both prediction files go to `MyDrive/DL-Project/result_finetune_annotated/instanovoplus/`.
Eval outputs (against annotated MGF) live under `result_finetune_annotated/`
to keep them separate from inference outputs (which go to
`result_finetune/<tool>/ecoli/...` for the FDR pipeline).


In [30]:
!mkdir -p /content/drive/MyDrive/DL-Project/result_finetune_annotated/instanovoplus
!cp /content/result_finetune_annotated/instanovoplus/Ecoli_EV_2.pretrained.csv /content/drive/MyDrive/DL-Project/result_finetune_annotated/instanovoplus/
!cp /content/result_finetune_annotated/instanovoplus/Ecoli_EV_2.finetuned.csv  /content/drive/MyDrive/DL-Project/result_finetune_annotated/instanovoplus/
!ls -lh /content/drive/MyDrive/DL-Project/result_finetune_annotated/instanovoplus/


total 730K
-rw------- 1 root root 365K May  2 18:02 Ecoli_EV_2.finetuned.csv
-rw------- 1 root root 366K May  2 18:02 Ecoli_EV_2.pretrained.csv
